# SALib: Sensitivity Analysis

## Overview
SALib (Sensitivity Analysis Library) provides:
- Sobol indices (global sensitivity)
- Morris method (screening)
- FAST method
- Saltelli method
- Delta method

## Why SALib for Engineering?
- Identify which parameters most affect CFD results
- Reduce model complexity
- Guide optimization efforts

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import salib
from salib import problem, sample, analyze

# 1. Define the Problem
print("=== Define the Problem ===")

# Define parameter bounds
problem_def = {
    'num_vars': 3,
    'names': ['Re', 'alpha', 'M'],
    'bounds': [[1e5, 1e6], [0, 20], [0.3, 0.9]]
}

print(f"Problem definition:")
print(f"Parameters: {problem_def['names']}")
print(f"Bounds: {problem_def['bounds']}")

# 2. Generate Samples (Sobol)
print("\n=== Generate Samples ===")

n_samples = 1024  # Must be a power of 2 for Sobol
param_values = sample.sobol(problem_def, n_samples, calc_second_order=True)

print(f"Sample shape: {param_values.shape}")
print(f"First 5 samples:\n{param_values[:5]}")

# 3. Evaluate the Model
print("\n=== Evaluate the Model ===")

def cfd_model(params):
    """
    Simple CFD model: drag coefficient
    params: [Re, alpha, M]
    """
    Re, alpha, M = params
    
    # Non-linear model with interactions
    cd = (0.02 + 
          0.01 * (Re / 1e6) + 
          0.005 * (alpha / 10)**2 + 
          0.02 * (M - 0.5)**2 +
          0.001 * (Re / 1e6) * (alpha / 10))
    
    return cd

# Evaluate model for all samples
Y = np.array([cfd_model(x) for x in param_values])

print(f"Model output range: [{Y.min():.4f}, {Y.max():.4f}]")
print(f"Model output mean: {Y.mean():.4f}")

# 4. Analyze Sensitivity
print("\n=== Analyze Sensitivity ===")

Si = analyze.sobol(problem_def, Y, calc_second_order=True)

print(f"First-order Sobol indices:")
for i, name in enumerate(problem_def['names']):
    print(f"  {name}: {Si['S1'][i]:.4f} ± {Si['S1_conf'][i]:.4f}")

print(f"\nTotal-order Sobol indices:")
for i, name in enumerate(problem_def['names']):
    print(f"  {name}: {Si['ST'][i]:.4f} ± {Si['ST_conf'][i]:.4f}")

# 5. Plot Results
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# First-order indices
names = problem_def['names']
S1 = Si['S1']
S1_conf = Si['S1_conf']

axes[0].barh(names, S1, xerr=S1_conf, capsize=5, color='steelblue')
axes[0].set_xlabel('First-Order Sobol Index')
axes[0].set_title('First-Order Sensitivity')
axes[0].grid(True, alpha=0.3, axis='x')

# Total-order indices
ST = Si['ST']
ST_conf = Si['ST_conf']

axes[1].barh(names, ST, xerr=ST_conf, capsize=5, color='coral')
axes[1].set_xlabel('Total-Order Sobol Index')
axes[1].set_title('Total-Order Sensitivity')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## Morris Method (Screening)

Faster screening method for identifying important parameters.

In [ ]:
# 6. Morris Method
print("\n=== Morris Method ===")

# Generate Morris samples
n_trajectories = 20
param_values_morris = sample.morris(problem_def, n_trajectories=n_trajectories)

print(f"Morris sample shape: {param_values_morris.shape}")

# Evaluate model
Y_morris = np.array([cfd_model(x) for x in param_values_morris])

# Analyze
Si_morris = analyze.morris(problem_def, param_values_morris, Y_morris)

print(f"\nMorris indices:")
for i, name in enumerate(problem_def['names']):
    mu_star = Si_morris['mu_star'][i]
    sigma = Si_morris['sigma'][i]
    print(f"  {name}: mu* = {mu_star:.4f}, sigma = {sigma:.4f}")

# Plot Morris results
plt.figure(figsize=(10, 6))
names = problem_def['names']
mu_star = Si_morris['mu_star']
sigma = Si_morris['sigma']

x = np.arange(len(names))
width = 0.35

plt.bar(x - width/2, mu_star, width, label='mu* (mean effect)', color='steelblue')
plt.bar(x + width/2, sigma, width, label='sigma (uncertainty)', color='coral')

plt.xlabel('Parameter')
plt.ylabel('Index')
plt.title('Morris Sensitivity Analysis')
plt.xticks(x, names)
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Engineering Application: CFD Parameter Screening

Identify which parameters most affect the drag coefficient.

In [ ]:
# 7. Engineering Application: Parameter Screening
print("\n=== Parameter Screening ===")

# Define a more complex model with 5 parameters
problem_def_5 = {
    'num_vars': 5,
    'names': ['Re', 'alpha', 'M', 't/c', 'roughness'],
    'bounds': [[1e5, 1e6], [0, 20], [0.3, 0.9], [0.05, 0.2], [0, 1e-6]]
}

def cfd_model_5(params):
    """5-parameter CFD model"""
    Re, alpha, M, tc, roughness = params
    
    cd = (0.02 + 
          0.01 * (Re / 1e6) + 
          0.005 * (alpha / 10)**2 + 
          0.02 * (M - 0.5)**2 +
          0.001 * (tc - 0.1)**2 +
          0.0001 * (roughness / 1e-6)**2)
    
    return cd

# Generate Sobol samples
n_samples = 1024
param_values = sample.sobol(problem_def_5, n_samples, calc_second_order=True)

# Evaluate model
Y = np.array([cfd_model_5(x) for x in param_values])

# Analyze
Si = analyze.sobol(problem_def_5, Y, calc_second_order=True)

# Plot results
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

names = problem_def_5['names']
S1 = Si['S1']
S1_conf = Si['S1_conf']

axes[0].barh(names, S1, xerr=S1_conf, capsize=5, color='steelblue')
axes[0].set_xlabel('First-Order Sobol Index')
axes[0].set_title('First-Order Sensitivity (5 Parameters)')
axes[0].grid(True, alpha=0.3, axis='x')

ST = Si['ST']
ST_conf = Si['ST_conf']

axes[1].barh(names, ST, xerr=ST_conf, capsize=5, color='coral')
axes[1].set_xlabel('Total-Order Sobol Index')
axes[1].set_title('Total-Order Sensitivity (5 Parameters)')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print(f"\nSensitivity ranking (by total-order index):")
sorted_indices = np.argsort(ST)[::-1]
for i in sorted_indices:
    print(f"  {names[i]}: {ST[i]:.4f}")

## Exercises

1. Perform Sobol sensitivity analysis for a 4-parameter model.
2. Compare Sobol vs Morris for the same problem.
3. Identify which parameters can be fixed to reduce model complexity.